# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to use the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library to access, explore, and process a [FAIR^2 Clinical CRC Survivor dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) specified by a Croissant schema.

### Dataset Source
- **FAIR^2 Croissant schema URL:** https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
- **Dataset title:** Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading

We'll load the dataset metadata and records with the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Croissant schema URL for the FAIR^2 CRC survivors dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}")

## 2. Data Overview

Let's review the available record sets, along with each entity's `@id` as defined by the Croissant schema. We'll also list the fields and their corresponding `@id`s for further reference.

In [ ]:
# List all record sets (tables) in the dataset using their @id
record_sets = []
if hasattr(metadata, "record_sets"):
    for rs in metadata.record_sets:
        print(f"Record Set name: {rs.name}\nRecord Set @id: {rs.id}")
        if hasattr(rs, "fields"):
            print("  Fields:")
            for field in rs.fields:
                print(f"    - {field.name} (@id: {field.id})")
        print()
        record_sets.append(rs.id)
else:
    print("No record sets found in the metadata.")

## 3. Data Extraction

Load the data records from the primary record set into a pandas DataFrame for analysis.

We'll use exact `@id` field values obtained from the metadata in the previous step to ensure unambiguous data access.

> _If you are not sure which record set(s) to use, choose the main record set containing clinical patient records._

In [ ]:
# For this dataset, there may only be one primary record set. We'll extract all record sets for demonstration.
import numpy as np

dataframes = {}
for record_set_id in record_sets:
    # Use the unique @id to load each record set
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"No records found for record set {record_set_id}")
        continue
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records from record set: {record_set_id}")

# View the columns (@id) of the primary record set DataFrame
if record_sets:
    main_record_set = record_sets[0]  # pick first if only one
    print("\nColumns (by field @id) in the main record set:")
    print(list(dataframes[main_record_set].columns))
    display(dataframes[main_record_set].head())
else:
    print("No record sets loaded.")

## 4. Exploratory Data Analysis (EDA)

Let's demonstrate basic data processing steps such as filtering records by field values, normalizing a numeric field, and grouping the data. 

You'll need to use the exact `@id` values for the fields. If you're not sure which are numeric, review the columns above for identifiers such as 'age', 'interval', or counts.

> _**Note:** If the field names in the DataFrame look unfamiliar, cross-reference with the fields listed in the metadata above._

In [ ]:
# Select a numeric field by its @id. Adjust this if needed based on your data overview above.
# Common numeric choices might include fields like 'age', 'diagnosis_interval', etc.
df = dataframes[main_record_set]
numeric_fields = [col for col in df.columns if df[col].dtype in [np.int64, np.float64, "int64", "float64"]]
if not numeric_fields:
    # Try inferring numeric columns
    numeric_fields = [col for col in df.columns if pd.to_numeric(df[col], errors='coerce').notnull().sum() > 0]
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Using numeric field @id: {numeric_field_id}")
else:
    raise Exception("No numeric field found for EDA. Please check the dataset.")

threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype != 'O' else 0 # set an appropriate threshold
filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()
print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    pd.to_numeric(filtered_df[numeric_field_id], errors="coerce") - pd.to_numeric(filtered_df[numeric_field_id], errors="coerce").mean()
) / pd.to_numeric(filtered_df[numeric_field_id], errors="coerce").std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to group by a likely categorical field
possible_cat_fields = [col for col in df.columns if col != numeric_field_id and df[col].nunique() < df.shape[0]/2]
if possible_cat_fields:
    group_field_id = possible_cat_fields[0]
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped means by {group_field_id}:")
    display(grouped_df.head())
else:
    print("No suitable categorical field to group by.")

## 5. Visualization

Let's visualize the distribution of our numeric field, and (optionally) compare its distribution across a categorical grouping variable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field
plt.figure(figsize=(7, 4))
sns.histplot(pd.to_numeric(df[numeric_field_id], errors="coerce").dropna(), bins=15)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If grouped by a field, plot boxplot for comparison
if 'group_field_id' in locals():
    plt.figure(figsize=(7, 4))
    sns.boxplot(x=df[group_field_id], y=pd.to_numeric(df[numeric_field_id], errors='coerce'))
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion

- Using the Croissant schema and `mlcroissant`, we easily loaded both the dataset's structure and its contents.
- We demonstrated core EDA steps: inspecting metadata, loading records, examining fields (by `@id`), filtering and normalizing data, and visualizing results.
- This workflow supports reproducible and FAIR data exploration for clinical and molecular data.

Refer to [mlcroissant documentation](https://github.com/mlcommons/croissant) for more advanced data wrangling, transformation, and schema-driven machine learning pipelines.